In [76]:
import polars as pl

In [77]:
fact = pl.read_parquet("../data/gold/fact_trafico_hora.parquet")
fecha = pl.read_parquet("../data/gold/dim_fecha.parquet")
sensor = pl.read_parquet("../data/gold/dim_sensor.parquet")

In [78]:
print(f"Fact: {fact.shape}")
print(f"Fecha: {fecha.shape}")
print(f"Sensor: {sensor.shape}")

Fact: (40521393, 13)
Fecha: (365, 8)
Sensor: (5088, 9)


In [79]:
fact.schema

Schema([('id_sensor', Int32),
        ('id_fecha', Date),
        ('hora', Int32),
        ('intensidad_media', Float64),
        ('intensidad_max', Float64),
        ('intensidad_min', Float64),
        ('ocupacion_media', Float64),
        ('ocupacion_max', Float64),
        ('velocidad_media', Float64),
        ('velocidad_min', Float64),
        ('num_mediciones', Int64),
        ('num_error_E', Float64),
        ('porcentaje_calidad', Float64)])

In [80]:
fecha.schema

Schema([('id_fecha', Date),
        ('año', Int64),
        ('mes', Int64),
        ('nombre_mes', String),
        ('trimestre', Int64),
        ('dia', Int64),
        ('dia_semana', Int64),
        ('fin_semana', Boolean)])

In [81]:
sensor.schema

Schema([('id_sensor', Int32),
        ('tipo_elem', String),
        ('distrito', Int32),
        ('cod_cent', String),
        ('nombre_norm', String),
        ('utm_x', Float64),
        ('utm_y', Float64),
        ('latitud', Float64),
        ('longitud', Float64)])

In [82]:
fact.head()

id_sensor,id_fecha,hora,intensidad_media,intensidad_max,intensidad_min,ocupacion_media,ocupacion_max,velocidad_media,velocidad_min,num_mediciones,num_error_E,porcentaje_calidad
i32,date,i32,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64
1001,2026-04-18,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1001,2026-04-21,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1001,2026-04-21,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1002,2026-04-16,10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1002,2026-04-17,21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0


In [83]:
fecha.head()

id_fecha,año,mes,nombre_mes,trimestre,dia,dia_semana,fin_semana
date,i64,i64,str,i64,i64,i64,bool
2026-04-17,2026,4,"""April""",2,17,5,false
2026-04-21,2026,4,"""April""",2,21,2,false
2026-04-23,2026,4,"""April""",2,23,4,false
2026-04-25,2026,4,"""April""",2,25,6,true
2026-04-11,2026,4,"""April""",2,11,6,true


In [84]:
sensor.head()

id_sensor,tipo_elem,distrito,cod_cent,nombre_norm,utm_x,utm_y,latitud,longitud
i32,str,i32,str,str,f64,f64,f64,f64
1013,"""other""",9,"""18XC46PM01""","""18xc46pm01""",438704.110899,4.4746e6,40.419971,-3.722532
1019,"""other""",12,"""13XL49PM01""","""13xl49pm01""",440866.291144,4.4710e6,40.387345,-3.696709
1020,"""other""",2,"""13NL60PM01""","""13nl60pm01""",440751.95328,4.4712e6,40.389482,-3.698078
1022,"""other""",12,"""14RR28PM01""","""14rr28pm01""",440316.937491,4.4717e6,40.393637,-3.703247
1031,"""other""",2,"""15RR60PM01""","""15rr60pm01""",439559.313876,4.4724e6,40.40015,-3.712242


In [85]:
print(f"Sensores fact: {fact['id_sensor'].n_unique()}")
print(f"Sensores dimensión: {sensor['id_sensor'].n_unique()}")

Sensores fact: 4933
Sensores dimensión: 5088


In [86]:
sensores_faltantes = (
    set(fact["id_sensor"].to_list())
    - set(sensor["id_sensor"].to_list())
)

print(len(sensores_faltantes))

0


In [87]:
df = (
    fact
    .join(fecha, on="id_fecha", how="left")
    .join(sensor, on="id_sensor", how="left")
)

In [88]:
print(fact.height)
print(df.height)

40521393
40521393


In [89]:
df.select(
    pl.col("año").is_null().sum()
)

año
u32
0


In [90]:
df.select(
    pl.col("tipo_elem").is_null().sum()
)

tipo_elem
u32
0


In [91]:
print(df.height)

print(
    df.select(
        pl.struct(
            ["id_sensor", "id_fecha", "hora"]
        ).n_unique()
    )
)

40521393


shape: (1, 1)
┌───────────┐
│ id_sensor │
│ ---       │
│ u32       │
╞═══════════╡
│ 40521393  │
└───────────┘


In [92]:
(
    df.null_count()
      .transpose(
          include_header=True,
          header_name="columna",
          column_names=["nulos"]
      )
      .sort("nulos", descending=True)
)

columna,nulos
str,u32
"""nombre_norm""",111090
"""distrito""",42529
"""velocidad_media""",33781
"""velocidad_min""",33781
"""id_sensor""",0
…,…
"""cod_cent""",0
"""utm_x""",0
"""utm_y""",0


In [93]:
df = df.drop(
    "nombre_norm",
    "cod_cent",
    "utm_x",
    "utm_y"
)

In [94]:
df = df.with_columns(
    (
        pl.col("tipo_elem") == "M30"
    )
    .cast(pl.Int8)
    .alias("es_m30")
)

In [95]:
df = df.with_columns(
    pl.col("distrito")
      .fill_null(0)
      .cast(pl.Int32)
)

In [96]:
df = df.with_columns([

    pl.when(pl.col("es_m30") == 0)
      .then(0)
      .otherwise(pl.col("velocidad_media"))
      .alias("velocidad_media"),

    pl.when(pl.col("es_m30") == 0)
      .then(0)
      .otherwise(pl.col("velocidad_min"))
      .alias("velocidad_min")

])

In [97]:
df = df.drop("tipo_elem")

In [98]:
(
    df.null_count()
      .transpose(
          include_header=True,
          header_name="columna",
          column_names=["nulos"]
      )
      .sort("nulos", descending=True)
)

columna,nulos
str,u32
"""velocidad_media""",33781
"""velocidad_min""",33781
"""id_sensor""",0
"""id_fecha""",0
"""hora""",0
…,…
"""fin_semana""",0
"""distrito""",0
"""latitud""",0


In [99]:
df.filter(
    pl.col("velocidad_media").is_null()
).group_by("es_m30").len()

es_m30,len
i8,u32
1,33781


In [100]:
df = df.drop_nulls(
    subset=[
        "velocidad_media",
        "velocidad_min"
    ]
)

In [101]:
(
    df.null_count()
      .transpose(
          include_header=True,
          header_name="columna",
          column_names=["nulos"]
      )
      .sort("nulos", descending=True)
)

columna,nulos
str,u32
"""id_sensor""",0
"""id_fecha""",0
"""hora""",0
"""intensidad_media""",0
"""intensidad_max""",0
…,…
"""fin_semana""",0
"""distrito""",0
"""latitud""",0


In [102]:
df.write_parquet("../data/gold/base_modelado.parquet")